In [52]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.stem import WordNetLemmatizer  
from nltk.tokenize import word_tokenize  
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [21]:
!pip install contractions


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
porterstemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()  

In [2]:
df = pd.read_csv(r"C:\Users\rajve\OneDrive\Desktop\NLP_PYTHON\NLP\all_kindle_review .csv")

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df=df[['reviewText', 'summary','rating']]

In [5]:
df.head()

,reviewText,summary,rating
0,"Jace Rankin may be short, but he's nothing to ...",Entertaining But Average,3
1,Great short read. I didn't want to put it dow...,Terrific menage scenes!,5
2,I'll start by saying this is the first of four...,Snapdragon Alley,3
3,Aggie is Angela Lansbury who carries pocketboo...,very light murder cozy,3
4,I did not expect this type of book to be in li...,Book,4


In [6]:
df.isnull().sum()

reviewText    0
summary       2
rating        0
dtype: int64

In [42]:
df.shape

(12000, 3)

In [18]:
stop_words=set(stopwords.words('english'))

In [19]:
negations = {"not", "no", "nor", "don't", "didn't", "won't", "wouldn't", 
             "shouldn't", "couldn't", "isn't", "wasn't", "aren't", "weren't", 
             "haven't", "hasn't", "hadn't", "can't"}
custom_stopwords = stop_words - negations

In [ ]:
custom_stopwords

In [24]:
# Import the contractions library to expand shortened words like "don't" to "do not"
import contractions

def cleaner(text, custom_stopwords=None, remove_numbers=False, keep_basic_punctuation=True):
    """
    Comprehensive review text cleaner
    
    Parameters:
    - text: Input text to clean
    - custom_stopwords: Optional set of custom stopwords to remove
    - remove_numbers: If True, removes all numbers from text
    - keep_basic_punctuation: If True, keeps basic punctuation like .?!,; otherwise removes all punctuation
    """
    
    # Initialize custom_stopwords as empty set if not provided
    if custom_stopwords is None:
        custom_stopwords = set()
    
    # Check for invalid input: if text is not string or empty/whitespace only
    if not isinstance(text, str) or text.strip() == "":
        return ""  # Return empty string for invalid input
    
    # ======================================================
    # TEXT CLEANING STEPS
    # ======================================================
    
    # Expand contractions: "don't" becomes "do not", "can't" becomes "cannot", etc.
    text = contractions.fix(text)
    
    # Remove URLs: matches http://, https://, or www. patterns
    text = re.sub(r'https?://\S+|www\.\S+', '', text) 
    
    # Remove HTML tags: matches anything between < and >
    text = re.sub(r'<.*?>', '', text)
    
    # Remove email addresses: matches pattern like user@domain.com
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove mentions and hashtags: matches @username or #hashtag
    text = re.sub(r'[@#]\w+', '', text)
    
    # Remove numbers if specified (optional)
    if remove_numbers:
        text = re.sub(r'\d+', '', text)  # \d+ matches one or more digits
    
    # Handle punctuation based on user preference
    if keep_basic_punctuation:
        # Keep only: word characters, whitespace, and basic punctuation (.?!,)
        # [^\w\s\.\?!,] means "remove everything except words, whitespace, and .?!,"
        text = re.sub(r'[^\w\s\.\?!,]', '', text)
    else:
        # Remove all non-alphabetic characters except spaces
        # [^a-zA-Z\s] means "remove everything except letters a-z and whitespace"
        text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Normalize whitespace: replace multiple spaces/tabs/newlines with single space
    text = re.sub(r'\s+', ' ', text).strip()  # \s+ matches one or more whitespace characters
    
    # Final check for empty text after all cleaning
    if text == "":  
        return ""
    
    # Convert entire text to lowercase for consistency
    text = text.lower()
    
    # ======================================================
    # TOKENIZATION AND PROCESSING
    # ======================================================
    
    # Split text into individual words (tokens)
    tokens = word_tokenize(text)
    
    # Process each token:
    # 1. Lemmatize: convert words to their base form (e.g., "running" → "run")
    # 2. Filter out stopwords (both default and custom)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in custom_stopwords]
    
    # Join the cleaned tokens back into a single string
    return ' '.join(tokens)

In [27]:
df1=df.copy()

In [28]:
# Apply the cleaning function
df1['cleaned_review'] = df['reviewText'].apply(
    lambda x: cleaner(
        x, 
        custom_stopwords=custom_stopwords,
        remove_numbers=True,           # Remove numbers
        keep_basic_punctuation=False   # Remove all punctuation
    )
)
df1['cleaned_summary'] = df['summary'].apply(
    lambda x: cleaner(
        x, 
        custom_stopwords=custom_stopwords,
        remove_numbers=True,           # Remove numbers
        keep_basic_punctuation=False   # Remove all punctuation
    )
)


In [31]:
df1.isnull().sum()

reviewText         0
summary            2
rating             0
cleaned_review     0
cleaned_summary    0
dtype: int64

In [32]:
df1=df1[['cleaned_review', 'cleaned_summary','rating']]

In [33]:
df1.head()

,cleaned_review,cleaned_summary,rating
0,jace rankin may short nothing mess man hauled ...,entertaining average,3
1,great short read not want put read one sitting...,terrific menage scene,5
2,start saying first four book not expecting con...,snapdragon alley,3
3,aggie angela lansbury carry pocketbook instead...,light murder cozy,3
4,not expect type book library pleased find pric...,book,4


In [34]:
df1['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [35]:
df2=df1.copy()
df2['rating'] = df2['rating'].apply(lambda x: 1 if x > 3 else 0)


In [36]:
df2.head()

,cleaned_review,cleaned_summary,rating
0,jace rankin may short nothing mess man hauled ...,entertaining average,0
1,great short read not want put read one sitting...,terrific menage scene,1
2,start saying first four book not expecting con...,snapdragon alley,0
3,aggie angela lansbury carry pocketbook instead...,light murder cozy,0
4,not expect type book library pleased find pric...,book,1


In [37]:
df3=df2.copy()
df4=df2.copy()
df5=df2.copy()

In [40]:
# Combine both into one text
df3['combined_summary_review'] = df3['cleaned_summary']+ " " + df3['cleaned_review']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df3['combined_summary_review'], 
    df3['rating'], 
    test_size=0.2, 
    random_state=42,
    stratify=df3['rating']  # optional, keeps class balance
)


In [41]:
df3.head()

,cleaned_review,cleaned_summary,rating,combined_summary_review
0,jace rankin may short nothing mess man hauled ...,entertaining average,0,entertaining average jace rankin may short not...
1,great short read not want put read one sitting...,terrific menage scene,1,terrific menage scene great short read not wan...
2,start saying first four book not expecting con...,snapdragon alley,0,snapdragon alley start saying first four book ...
3,aggie angela lansbury carry pocketbook instead...,light murder cozy,0,light murder cozy aggie angela lansbury carry ...
4,not expect type book library pleased find pric...,book,1,book not expect type book library pleased find...


In [44]:
df3=df3[['combined_summary_review','rating']]

In [45]:
df3.head()

,combined_summary_review,rating
0,entertaining average jace rankin may short not...,0
1,terrific menage scene great short read not wan...,1
2,snapdragon alley start saying first four book ...,0
3,light murder cozy aggie angela lansbury carry ...,0
4,book not expect type book library pleased find...,1


In [47]:
bow=CountVectorizer()
tfidf=TfidfVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [ ]:

nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)
mn_model_bow=MultinomialNB().fit(X_train_bow,y_train)
mn_model_tfidf=MultinomialNB().fit(X_train_tfidf,y_train)
rf_model_bow=RandomForestClassifier().fit(X_train_bow,y_train)
rf_model_tfidf=RandomForestClassifier().fit(X_train_tfidf,y_train)

In [54]:
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)

In [55]:
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [56]:
mn_model_bow=MultinomialNB().fit(X_train_bow,y_train)

In [57]:
mn_model_tfidf=MultinomialNB().fit(X_train_tfidf,y_train)

In [66]:
rf_model_bow=RandomForestClassifier().fit(X_train_bow,y_train)

In [ ]:
rf_model_tfidf=RandomForestClassifier().fit(X_train_tfidf,y_train)

In [59]:
# nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
y_pred_bow_nb=nb_model_bow.predict(X_test_bow)
print("BOW ACCURACY WITH NB :",accuracy_score(y_test,y_pred_bow_nb))


BOW ACCURACY WITH NB : 0.6229166666666667


In [ ]:
# nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)
# y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)
# print("tfidf ACCURACY :",accuracy_score(y_test,y_pred_tfidf))

In [60]:
y_pred_bow_mn=mn_model_bow.predict(X_test_bow)
print("BOW ACCURACY WITH MULTINOMIAL :",accuracy_score(y_test,y_pred_bow_mn))


BOW ACCURACY WITH MULTINOMIAL : 0.8516666666666667


In [ ]:
y_pred_bow_rf=rf_model_bow.predict(X_test_bow)
print("BOW ACCURACY WITH RANDOM FOREST CLASSIFIER :",accuracy_score(y_test,y_pred_bow_rf))


In [61]:
y_pred_tfidf_nb=nb_model_tfidf.predict(X_test_tfidf)
print("TFIDF ACCURACY WITH NB :",accuracy_score(y_test,y_pred_tfidf_nb))


TFIDF ACCURACY WITH NB : 0.6508333333333334


In [62]:
y_pred_tfidf_mn=mn_model_tfidf.predict(X_test_tfidf)
print("TFIDF ACCURACY WITH MULTINOMIAL :",accuracy_score(y_test,y_pred_tfidf_mn))


TFIDF ACCURACY WITH MULTINOMIAL : 0.8525
